In [65]:
import pandas as pd
import numpy as np
import pickle

#### Open artist with country information

In [66]:
artistCountries = pd.read_csv("Data\\artists.csv", low_memory=False)

In [67]:
artistCountries = artistCountries[["artist_mb", "country_mb",]]

In [68]:
artistCountries = artistCountries.dropna()

In [69]:
artistCountries


,artist_mb,country_mb
0,Coldplay,United Kingdom
1,Radiohead,United Kingdom
2,Red Hot Chili Peppers,United States
3,Rihanna,United States
4,Eminem,United States
...,...,...
1466078,정은지,South Korea
1466079,남태현,South Korea
1466080,헤일로,South Korea
1466081,서현진,South Korea


#### Remove duplicates

In [70]:
# Found duplicate artists, so will only count the first one, since it is sorted by a popularity rating
artistCountries[artistCountries["artist_mb"] == "Queen"]

,artist_mb,country_mb
9,Queen,United Kingdom
1266828,Queen,United States
1266829,Queen,Vietnam
1461165,Queen,Japan


In [71]:
artistCountries.drop_duplicates(subset=["artist_mb"], inplace=True)

In [72]:
artistCountries[artistCountries["artist_mb"] == "Queen"]

,artist_mb,country_mb
9,Queen,United Kingdom


#### Create dictionaries to convert the countries to ISO codes

In [73]:
# Making dictionary to converet ISO-alpha2 codes to their ISO-alpha3 codes
countryInfo = pd.read_csv("Data\\CountryInfo.csv")
alpha2to3 = {}
for _, row in countryInfo[["alpha-2", "alpha-3"]].iterrows():
    alpha2to3[row["alpha-2"]] = row["alpha-3"]
    
# A few codes were not included in the dataset
alpha2to3["NA"] = "NAM"
alpha2to3["AN"] = "ANT"
alpha2to3["YU"] = "YUG"
# Kosovo is not officialy listed as ISO country, this is an unofficial code
alpha2to3["XK"] = "XKX"
with open("Data\\Vars\\alpha2to3.pkl", "wb+") as f:
    pickle.dump(alpha2to3, f)

In [74]:
# Making dictionaries to convert country to their ISO-alpha2 codes
codesCountry = {}
countryCodes = {}
with open("Data\\countryCodes.txt", "r") as f:
    file = f.read()
    for c in file.split("\n")[1:]:
        cc = c.split(",")
        codesCountry[cc[1]] = cc[0]
        countryCodes[cc[0]] = cc[1]
        

In [75]:
countryInfo[countryInfo["name"] == "Kosovo"]

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code


In [76]:
# Dictionaries to convert ISO-alpha3 codes to their full name and vice versa
iso3Country = {}
countryISO3 = {}

for country in countryCodes.keys():
    iso3Country[alpha2to3[countryCodes[country]]] = country
    countryISO3[country] = alpha2to3[countryCodes[country]]
    

with open("Data\\Vars\\iso3Country.pkl", "wb+") as f:
    pickle.dump(iso3Country, f)
    
with open("Data\\Vars\\countryISO3.pkl", "wb+") as f:
    pickle.dump(countryISO3, f)

In [77]:
# Found some artist that don't have a code, so in the underneath cells try to remedy that manually 
noCode = []
for country in artistCountries["country_mb"].unique():
    if country not in countryCodes.keys():
        noCode.append(country)

In [78]:
print(noCode)

['Soviet Union', 'Europe', 'East Germany', 'Czechoslovakia', 'Northern Cyprus', 'Serbia and Montenegro', 'Bonaire, Sint Eustatius and Saba', 'Orcas Island']


In [79]:
# Adding some variations and none if no code could be found
newCodes = [ ["Soviet Union", "RU"], 
            ["Europe", "None"],
            ["East Germany", "DE"],
            ["Czechoslovakia", "None"], 
            ["Northern Cyprus", "CY"], 
            ["Serbia and Montenegro", "None"], 
            ["Bonaire, Sint Eustatius and Saba", "BQ"],
            ["Orcas Island", "None"]]

In [80]:
for country, code in newCodes:
    countryCodes[country] = code

#### Add new information to the table

In [81]:
# Make ISO-2 row
artistCountry = []
for i, row in artistCountries.iterrows():
    artistCountry.append(countryCodes[row["country_mb"]])

In [82]:
len(artistCountry)

611834

In [83]:
artistCountries["ISO-2"] = artistCountry

In [84]:
# Origins that will not be included 
artistCountries[artistCountries["ISO-2"] == "None"]["country_mb"].unique()

array(['Europe', 'Czechoslovakia', 'Serbia and Montenegro',
       'Orcas Island'], dtype=object)

In [85]:
# All artists that don't have country information
artistCountries[artistCountries["ISO-2"] == "None"]["artist_mb"].unique()

array(['OST', 'Storm Corrosion', 'Tasha', 'Trees of Eternity',
       'Black Attack', 'Bosco Rogers', 'BOTH', 'Tzolk’in', 'Enshine',
       'The Matadors', 'Exit Eden', 'Josef Suk', 'TriORE',
       'Andy Sheppard Quartet', 'Charlie Straight', 'Franz Kafka',
       'Life is Pain', 'Servants of the Apocalyptic Goat Rave',
       'Eriq Johnson', 'Baba Yaga', 'Binary Park', 'Árstíðir lífsins',
       'Deep-pression', 'Memories of Machines', 'D.D. Sound',
       'Photophobia', 'Northward', 'Chamber Choir of Europe',
       'Love & Kisses', 'The Hard Way', 'Damh', 'Goga Sekulic',
       'Ugandan Methods', 'Layup', 'Mergel Kratzer', 'Kombat Unit',
       'Ominus', 'Ningizzia', 'The Birthday Girls', 'Rollàn', 'Dødsfall',
       'Diversidad', 'Timezone', 'Karel Ančerl', 'Heavyplastic',
       'Marsyas', 'Andre Rizo', 'Silente', 'Pherato', 'Andrey Pushkarev',
       'Maïa Vidal', 'Dataminions', 'Judita Čeřovská', 'Sembler Deah',
       'Jean-Pierre Bertrand', 'Eudaimony', 'Menneskerhat', 'Juven

In [86]:
artistCountries = artistCountries[artistCountries["ISO-2"] != "None"]
artistCountries

,artist_mb,country_mb,ISO-2
0,Coldplay,United Kingdom,GB
1,Radiohead,United Kingdom,GB
2,Red Hot Chili Peppers,United States,US
3,Rihanna,United States,US
4,Eminem,United States,US
...,...,...,...
1466078,정은지,South Korea,KR
1466079,남태현,South Korea,KR
1466080,헤일로,South Korea,KR
1466081,서현진,South Korea,KR


In [87]:
artistCountries["ISO-3"] = artistCountries["ISO-2"].map(alpha2to3)

C:\Users\tmjva\AppData\Local\Temp\ipykernel_21044\1080955536.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  artistCountries["ISO-3"] = artistCountries["ISO-2"].map(alpha2to3)


#### Final table

In [88]:
artistCountries

,artist_mb,country_mb,ISO-2,ISO-3
0,Coldplay,United Kingdom,GB,GBR
1,Radiohead,United Kingdom,GB,GBR
2,Red Hot Chili Peppers,United States,US,USA
3,Rihanna,United States,US,USA
4,Eminem,United States,US,USA
...,...,...,...,...
1466078,정은지,South Korea,KR,KOR
1466079,남태현,South Korea,KR,KOR
1466080,헤일로,South Korea,KR,KOR
1466081,서현진,South Korea,KR,KOR


In [89]:
artistCountries.to_csv("Data\\ArtistCountry.csv")